# §13.4.6 — 학습 길이를 넘긴 외삽 성능의 비교

> 딥러닝 교재 · 3부 13장 4절 6항 (🐍)
> 선행: §13.4.1(통일 표기) · §13.4.3(RoPE) · §13.4.4(ALiBi) · §13.4.7(교란 요인)

## 이 노트북이 답하는 질문

1. **학습 길이 $T_{\rm tr}$을 넘겨 평가하면 네 부호화는 각각 어떻게 열화하는가?**
2. **열화는 어느 위치에서 일어나는가?** 위치별 손실로 부검한다.
3. **RoPE의 위치 보간은 이 과제에서 도움이 되는가?** 처방이 진단에 조건부임을 확인한다.

**예상 실행 시간** CPU 약 6분 (`FAST = True`이면 약 2분).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제 — 조회 거리가 고정되지 않은 유도 문법

데이터는 다음 규칙으로 생성한다. 새 토큰이 등장할 때 그 후속 토큰이 무작위로 정해져
사전에 기록되고, **같은 토큰이 다시 나타나면 후속은 반드시 사전과 같다.** 따라서
재출현 토큰의 다음 토큰은 "이전 출현을 찾아 그 다음을 베끼는" 유도 헤드
\mycmt{induction head} 회로로 완전히 예측되고, 손실은 그 예측 가능 위치에서만 잰다.

설계 통제 하나가 결정적이다. 예비 실험에서 "앞 절반을 통째로 복제"하는 단순한
과제를 썼더니, 모델은 내용 조회 대신 **고정 오프셋 복사**(위치 $t$는 $t-T/2$를 본다)
라는 지름길을 배웠고, 길이를 늘리면 오프셋이 틀어져 네 부호화 모두 균등 분포보다
나쁜 손실로 무너졌다 — 부호화가 아니라 과제가 만든 인공물이다. 위 문법은 재출현
간격이 무작위이므로 그 지름길이 없다. **외삽 실험은 과제가 위치 지름길을 허용하는지
부터 검사해야 한다**(§13.4.7).

모델은 공용 미니 트랜스포머(2층, $h=4$, $d=48$)다.

In [ ]:
# ── 공용 미니 트랜스포머 (NumPy, 완전한 순전파+역전파) ──────────
# 구조: 임베딩 → L × [Pre-LN 블록 (MHA + MLP, 잔차)] → LN → 판독
# 위치 부호화: 'learned' | 'sin' | 'rope' | 'alibi' | 'none'

def make_config(V, d=48, L=2, h=4, T_max=64, pe='learned', causal=True, seed=0):
    dh = d // h
    rn = np.random.default_rng(seed)
    p = {}
    p['emb'] = rn.standard_normal((V, d)) * 0.5 / np.sqrt(d)
    if pe == 'learned':
        p['pos'] = rn.standard_normal((T_max, d)) * 0.5 / np.sqrt(d)
    for l in range(L):
        s = f'l{l}_'
        for nm in ['wq', 'wk', 'wv', 'wo']:
            p[s + nm] = rn.standard_normal((d, d)) / np.sqrt(d)
        p[s + 'ln1g'] = np.ones(d); p[s + 'ln1b'] = np.zeros(d)
        p[s + 'w1'] = rn.standard_normal((d, 4 * d)) / np.sqrt(d)
        p[s + 'b1'] = np.zeros(4 * d)
        p[s + 'w2'] = rn.standard_normal((4 * d, d)) / np.sqrt(4 * d)
        p[s + 'b2'] = np.zeros(d)
        p[s + 'ln2g'] = np.ones(d); p[s + 'ln2b'] = np.zeros(d)
    p['lnfg'] = np.ones(d); p['lnfb'] = np.zeros(d)
    p['out'] = rn.standard_normal((d, V)) / np.sqrt(d)
    cfg = dict(V=V, d=d, L=L, h=h, dh=dh, T_max=T_max, pe=pe, causal=causal)
    return p, cfg

def _sin_pe(T, d):
    pos = np.arange(T)[:, None]
    l2 = np.arange(0, d, 2)[None, :]
    ang = pos / (10000.0 ** (l2 / d))
    pe = np.zeros((T, d))
    pe[:, 0::2] = np.sin(ang); pe[:, 1::2] = np.cos(ang)
    return pe

def _rope_angles(T, dh, scale=1.0):
    pos = np.arange(T)[:, None] * scale
    l2 = np.arange(0, dh, 2)[None, :]
    return pos / (10000.0 ** (l2 / dh))          # (T, dh/2)

def _rope_apply(x, ang, inverse=False):
    # x: (B,h,T,dh) — 짝수/홀수 쌍을 각도 ang(T,dh/2)만큼 회전
    c, s = np.cos(ang), np.sin(ang)
    if inverse:
        s = -s
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return np.stack([x1 * c - x2 * s, x1 * s + x2 * c], axis=-1).reshape(x.shape)

def _alibi_slopes(h):
    return np.array([2.0 ** (-8.0 * (i + 1) / h) for i in range(h)])

def _ln_f(x, g, b):
    mu = x.mean(-1, keepdims=True)
    xc = x - mu
    var = (xc ** 2).mean(-1, keepdims=True)
    inv = 1.0 / np.sqrt(var + 1e-5)
    xh = xc * inv
    return xh * g + b, (xh, inv)

def _ln_b(dy, cache, g):
    xh, inv = cache
    dxh = dy * g
    dg = (dy * xh).sum(axis=tuple(range(dy.ndim - 1)))
    db = dy.sum(axis=tuple(range(dy.ndim - 1)))
    dx = inv * (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True))
    return dx, dg, db

def forward(p, cfg, idx, targets=None, rope_scale=1.0, head_mask=None,
            skip=None, want_attn=False, kv_keep=None):
    """idx:(B,T) 정수. targets:(B,T) 또는 None.
    head_mask:(L,h) 0/1, skip: {'attn':set(l), 'mlp':set(l)},
    kv_keep:(T,) bool — 열 s의 키·값 사용 여부(캐시 축출 흉내)."""
    B, T = idx.shape
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    x = p['emb'][idx]                              # (B,T,d)
    if cfg['pe'] == 'learned':
        x = x + p['pos'][:T]
    elif cfg['pe'] == 'sin':
        x = x + _sin_pe(T, d)
    cache = {'idx': idx, 'T': T, 'B': B, 'xs': [], 'attn': []}
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    if cfg['causal']:
        nmask = np.triu(np.full((T, T), -np.inf), k=1)
    else:
        nmask = np.zeros((T, T))
    if kv_keep is not None:
        nmask = nmask.copy()
        nmask[:, ~kv_keep] = -np.inf
    if cfg['pe'] == 'alibi':
        sl = _alibi_slopes(h)
        dist = np.maximum(np.arange(T)[:, None] - np.arange(T)[None, :], 0)
        abias = -sl[:, None, None] * dist[None]    # (h,T,T)
    else:
        abias = np.zeros((1, T, T))
    skip = skip or {'attn': set(), 'mlp': set()}
    attns = []
    def split(z):
        return z.reshape(B, T, h, dh).transpose(0, 2, 1, 3)       # (B,h,T,dh)
    for l in range(L):
        s = f'l{l}_'
        c = {}
        if l not in skip['attn']:
            # ── MHA 가지 ──
            h1, c['ln1'] = _ln_f(x, p[s + 'ln1g'], p[s + 'ln1b'])
            c['h1'] = h1
            q = split(h1 @ p[s + 'wq']); k = split(h1 @ p[s + 'wk']); v = split(h1 @ p[s + 'wv'])
            if cfg['pe'] == 'rope':
                q = _rope_apply(q, ang); k = _rope_apply(k, ang)
            e = np.einsum('bhtd,bhsd->bhts', q, k) / np.sqrt(dh) + nmask + abias[None]
            e -= e.max(-1, keepdims=True)
            a = np.exp(e); a /= a.sum(-1, keepdims=True)
            if head_mask is not None:
                hm = head_mask[l][None, :, None, None]
            else:
                hm = 1.0
            av = np.einsum('bhts,bhsd->bhtd', a, v) * hm
            avm = av.transpose(0, 2, 1, 3).reshape(B, T, d)
            x = x + avm @ p[s + 'wo']
            c.update(q=q, k=k, v=v, a=a, avm=avm, hm=hm)
            attns.append(a)
        else:
            attns.append(None)
        if l not in skip['mlp']:
            # ── MLP 가지 ──
            h2, c['ln2'] = _ln_f(x, p[s + 'ln2g'], p[s + 'ln2b'])
            z1 = h2 @ p[s + 'w1'] + p[s + 'b1']
            r = np.maximum(z1, 0.0)               # ReLU (역전파 단순화)
            x = x + r @ p[s + 'w2'] + p[s + 'b2']
            c['mlp'] = (h2, z1, r)
        else:
            c['mlp'] = None
        cache[s] = c
    hf, cache['lnf'] = _ln_f(x, p['lnfg'], p['lnfb'])
    cache['hf'] = hf
    logits = hf @ p['out']
    cache['logits'] = logits
    out = {'logits': logits}
    if want_attn:
        out['attn'] = attns
    if targets is not None:
        valid = targets >= 0                      # -1 = 손실에서 제외
        tsafe = np.maximum(targets, 0)
        z = logits - logits.max(-1, keepdims=True)
        lse = np.log(np.exp(z).sum(-1))
        ll = z[np.arange(B)[:, None], np.arange(T)[None, :], tsafe] - lse
        out['loss'] = -(ll * valid).sum() / max(valid.sum(), 1)
        P = np.exp(z); P /= P.sum(-1, keepdims=True)
        cache['P'] = P; cache['targets'] = tsafe; cache['valid'] = valid
    out['cache'] = cache
    return out

def backward(p, cfg, cache, rope_scale=1.0):
    B, T = cache['B'], cache['T']
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    g = {k: np.zeros_like(v) for k, v in p.items()}
    P, targets, valid = cache['P'], cache['targets'], cache['valid']
    dlogits = P.copy()
    dlogits[np.arange(B)[:, None], np.arange(T)[None, :], targets] -= 1.0
    dlogits *= valid[:, :, None]
    dlogits /= max(valid.sum(), 1)
    hf = cache['hf']
    g['out'] = np.einsum('btd,btv->dv', hf, dlogits)
    dhf = dlogits @ p['out'].T
    dx, g['lnfg'], g['lnfb'] = _ln_b(dhf, cache['lnf'], p['lnfg'])
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    for l in range(L - 1, -1, -1):
        s = f'l{l}_'
        c = cache[s]
        if c['mlp'] is not None:
            h2, z1, r = c['mlp']
            dmlp = dx                                   # 잔차: 가지로 흘러드는 기울기
            g[s + 'w2'] += np.einsum('btf,btd->fd', r, dmlp)
            g[s + 'b2'] += dmlp.sum((0, 1))
            dr = dmlp @ p[s + 'w2'].T
            dz1 = dr * (z1 > 0)
            g[s + 'w1'] += np.einsum('btd,btf->df', h2, dz1)
            g[s + 'b1'] += dz1.sum((0, 1))
            dh2 = dz1 @ p[s + 'w1'].T
            dxi, dg2, db2 = _ln_b(dh2, c['ln2'], p[s + 'ln2g'])
            g[s + 'ln2g'] += dg2; g[s + 'ln2b'] += db2
            dx = dx + dxi
        if 'a' not in c:
            continue
        # MHA 가지
        dattn_out = dx
        g[s + 'wo'] += np.einsum('btd,bte->de', c['avm'], dattn_out)
        davm = dattn_out @ p[s + 'wo'].T
        dav = davm.reshape(B, T, h, dh).transpose(0, 2, 1, 3) * c['hm']
        a, q, k, v = c['a'], c['q'], c['k'], c['v']
        da = np.einsum('bhtd,bhsd->bhts', dav, v)
        dv = np.einsum('bhts,bhtd->bhsd', a, dav)
        de = a * (da - (a * da).sum(-1, keepdims=True))
        dq = np.einsum('bhts,bhsd->bhtd', de, k) / np.sqrt(dh)
        dk = np.einsum('bhts,bhtd->bhsd', de, q) / np.sqrt(dh)
        if cfg['pe'] == 'rope':
            dq = _rope_apply(dq, ang, inverse=True)
            dk = _rope_apply(dk, ang, inverse=True)
        def merge(z):
            return z.transpose(0, 2, 1, 3).reshape(B, T, d)
        dq, dk, dv = merge(dq), merge(dk), merge(dv)
        h1 = c['h1']
        g[s + 'wq'] += np.einsum('btd,bte->de', h1, dq)
        g[s + 'wk'] += np.einsum('btd,bte->de', h1, dk)
        g[s + 'wv'] += np.einsum('btd,bte->de', h1, dv)
        dh1 = dq @ p[s + 'wq'].T + dk @ p[s + 'wk'].T + dv @ p[s + 'wv'].T
        dxi, dg1, db1 = _ln_b(dh1, c['ln1'], p[s + 'ln1g'])
        g[s + 'ln1g'] += dg1; g[s + 'ln1b'] += db1
        dx = dx + dxi
    if cfg['pe'] == 'learned':
        g['pos'][:T] += dx.sum(0)
    np.add.at(g['emb'], cache['idx'], dx)
    return g

def adam_init(p):
    return {k: np.zeros_like(v) for k, v in p.items()}, {k: np.zeros_like(v) for k, v in p.items()}

def adam_step(p, g, m, v, t, lr=3e-3):
    for k in p:
        m[k] = 0.9 * m[k] + 0.1 * g[k]
        v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2
        p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)

def train_lm(p, cfg, sample_batch, steps, lr=3e-3, log_every=0, rope_scale=1.0):
    m, v = adam_init(p)
    hist = []
    for t in range(1, steps + 1):
        idx, tgt = sample_batch()
        out = forward(p, cfg, idx, targets=tgt, rope_scale=rope_scale)
        g = backward(p, cfg, out['cache'], rope_scale=rope_scale)
        adam_step(p, g, m, v, t, lr)
        hist.append(out['loss'])
        if log_every and t % log_every == 0:
            print(f"  step {t}: loss {np.mean(hist[-log_every:]):.3f}")
    return hist

In [ ]:
V = 64
T_TR = 48

def gen_batch(B, T, rn):
    idx = np.zeros((B, T), int)
    pred = np.zeros((B, T), bool)
    for b in range(B):
        succ = {}; seq = [1]; predictable = [False]; prev = 1
        while len(seq) < T:
            if prev in succ:                      # 후속이 사전에 있으면 강제
                nxt = succ[prev]; predictable.append(True)
            else:
                if succ and rn.random() < 0.55:   # 재방문 (다음 스텝이 예측 가능해짐)
                    nxt = int(rn.choice(list(succ.keys())))
                else:                             # 새 토큰
                    nxt = int(rn.integers(2, V))
                predictable.append(False)
                succ[prev] = nxt
            seq.append(nxt); prev = nxt
        idx[b] = seq; pred[b] = predictable
    tgt = np.full((B, T), -1)
    tgt[:, :-1] = np.where(pred[:, 1:], idx[:, 1:], -1)   # 예측 가능 위치만 손실
    return idx, tgt

PES = ['learned', 'sin', 'rope', 'alibi']
T_MAX_EVAL = 384
STEPS = 150 if FAST else 350
models = {}
for pe in PES:
    p, cfg = make_config(V=V, d=48, L=2, h=4, T_max=T_TR, pe=pe, seed=7)
    rn = np.random.default_rng(100)
    train_lm(p, cfg, lambda: gen_batch(32, T_TR, rn), steps=STEPS, lr=3e-3)
    if pe == 'learned':   # 학습 길이 밖 위치는 정의 자체가 없다 — 마지막 벡터를 재사용하는 응급 수선
        p['pos'] = np.concatenate([p['pos'], np.tile(p['pos'][-1:], (T_MAX_EVAL - T_TR, 1))])
        cfg['T_max'] = T_MAX_EVAL
    models[pe] = (p, cfg)
    idx, tgt = gen_batch(128, T_TR, np.random.default_rng(9))
    print(f"{pe:8s}: 학습 길이 손실 {forward(p, cfg, idx, targets=tgt)['loss']:.3f}"
          f"  ({time.time()-_t0:.0f}초)")

---
## 2. 길이를 훑는다 — 학습은 전부 $T_{\rm tr}=48$

In [ ]:
T_EVALS = [48, 96, 192] if FAST else [48, 96, 192, 384]
T_POS = 192
loss_curve = {pe: [] for pe in PES}
ent_curve = {pe: [] for pe in PES}
pos_loss = {}
for pe in PES:
    p, cfg = models[pe]
    for T in T_EVALS:
        idx, tgt = gen_batch(96, T, np.random.default_rng(70 + T))
        out = forward(p, cfg, idx, targets=tgt, want_attn=True)
        loss_curve[pe].append(out['loss'])
        A = out['attn'][-1]
        ent_curve[pe].append(-(A * np.log(A + 1e-12)).sum(-1)[:, :, T // 2:].mean())
        if T == T_POS:
            P = out['cache']['P']
            ll = -np.log(P[np.arange(96)[:, None], np.arange(T)[None, :],
                           np.maximum(tgt, 0)] + 1e-12)
            valid = (tgt >= 0)
            cnt = valid.sum(0).astype(float)
            pl = np.where(cnt > 0, (ll * valid).sum(0) / np.maximum(cnt, 1), np.nan)
            pos_loss[pe] = pl
    rel = loss_curve[pe][-1] / loss_curve[pe][0]
    print(f"{pe:8s}: 길이별 손실 {np.round(loss_curve[pe], 2)}  (열화 배율 ×{rel:.1f})")

---
## 3. RoPE 위치 보간 — 처방은 진단에 조건부다

회전 각도를 $T_{\rm tr}/T$배로 압축하는 위치 보간(§13.4.8)을 미세조정 없이 적용해 본다.

In [ ]:
p, cfg = models['rope']
rope_pi = []
for T in T_EVALS:
    idx, tgt = gen_batch(96, T, np.random.default_rng(70 + T))
    rope_pi.append(forward(p, cfg, idx, targets=tgt,
                           rope_scale=min(1.0, T_TR / T))['loss'])
print("RoPE 원본:", np.round(loss_curve['rope'], 2))
print("RoPE 보간:", np.round(rope_pi, 2))

---
## 4. 교재 그림 — fig_13_4_6

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()
COLS = {'learned': CB[4], 'sin': CB[1], 'rope': CB[5], 'alibi': CB[3]}
NAMES = {'learned': lab('학습 절대(+수선)', 'learned(+fix)'), 'sin': lab('사인', 'sinusoidal'),
         'rope': 'RoPE', 'alibi': 'ALiBi'}

# (a) 길이별 손실
ax = axes[0]
for pe in PES:
    ax.plot(T_EVALS, loss_curve[pe], 'o-', color=COLS[pe], ms=5, label=NAMES[pe])
ax.axvline(T_TR, color='k', lw=0.8, ls=':')
ax.text(T_TR + 4, ax.get_ylim()[1] * 0.93, lab('학습 길이', '$T_{tr}$'), fontsize=8)
ax.set_xlabel(lab('평가 길이 $T$', 'eval length'))
ax.set_ylabel(lab('손실 (예측 가능 위치)', 'loss'))
ax.set_title(lab('(a) 외삽 — 절대 손실과 열화 배율을 나눠 읽을 것', '(a) extrapolation'), fontsize=10)
ax.legend(fontsize=8)

# (b) 위치별 손실 (T=192)
ax = axes[1]
for pe in PES:
    ax.plot(np.arange(T_POS), pos_loss[pe], '-', color=COLS[pe], lw=1.1, label=NAMES[pe])
ax.axvline(T_TR, color='k', lw=0.8, ls=':')
ax.text(T_TR + 4, ax.get_ylim()[1] * 0.9, lab('학습 길이', '$T_{tr}$'), fontsize=8)
ax.set_xlabel(lab('위치 $t$', 'position'))
ax.set_ylabel(lab('위치별 손실', 'per-position loss'))
ax.set_title(lab(f'(b) 열화의 자리 ($T={T_POS}$)', '(b) per-position loss'), fontsize=10)
ax.legend(fontsize=8)

# (c) 어텐션 엔트로피의 이동
ax = axes[2]
for pe in PES:
    ax.plot(T_EVALS, ent_curve[pe], 'o-', color=COLS[pe], ms=5, label=NAMES[pe])
ax.set_xlabel(lab('평가 길이 $T$', 'eval length'))
ax.set_ylabel(lab('마지막 층 어텐션 엔트로피', 'attn entropy'))
ax.set_title(lab('(c) 길이가 늘면 분포 자체가 이동한다', '(c) entropy drift'), fontsize=10)
ax.legend(fontsize=8)

# (d) RoPE 보간
ax = axes[3]
ax.plot(T_EVALS, loss_curve['rope'], 'o-', color=CB[5], ms=5, label=lab('RoPE 원본', 'RoPE'))
ax.plot(T_EVALS, rope_pi, 's--', color=CB[6], ms=5, label=lab('RoPE + 위치 보간', 'RoPE + PI'))
ax.axvline(T_TR, color='k', lw=0.8, ls=':')
ax.set_xlabel(lab('평가 길이 $T$', 'eval length'))
ax.set_ylabel(lab('손실 (예측 가능 위치)', 'loss'))
ax.set_title(lab('(d) 밀 붕괴가 없으면 보간은 대가만 남긴다', '(d) interpolation'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_13_4_6')
plt.show()

> ### 읽는 법
>
> (a) 순위는 잣대에 따라 다르다. **절대 손실**은 RoPE가 전 구간 최저이고, **열화
> 배율**(자기 기준 대비)은 ALiBi가 최소다 — 거리 벌점이 길이를 기억하지 않으므로
> 곡선이 거의 수평이다(§13.4.4의 예측). 사인은 절대·배율 모두 나쁘고, 학습 절대는
> "마지막 위치 벡터 재사용"이라는 응급 수선 덕에 실행은 되지만 배율이 가장 크다 —
> 그리고 수선 없이는 **실행조차 되지 않는다**는 사실이 이 방식의 본질적 한계다.
> (b) 열화는 학습 길이 이후 위치에 집중된다. (c) 후보 수 증가에 따른 엔트로피 이동은
> 네 곡선 모두에 겹쳐 있는 공통 교란이다(§13.4.7).
> (d) 이 과제의 RoPE는 애초에 완만하게 열화하므로, 각도를 압축하는 보간은 **밀어낼
> 붕괴가 없고 고주파 분해능의 손실만 남긴다** — 원본보다 나빠진다. 보간이 유효한
> 것은 "각도 범위 밖"이 실패의 주원인일 때뿐이다. 처방은 진단에 조건부라는 §13.4.8의
> 경고가 실측으로 확인된 셈이다.

---
## 5. 자기 점검

1. 1절의 실패담(고정 오프셋 지름길)을 직접 재현해 보라. 복제 과제로 바꾸면 손실이 균등 분포($\log 62\approx4.1$)보다 왜 **더 나빠지는지** 설명하라.
2. ALiBi의 헤드별 기울기(`_alibi_slopes`)에서 가장 완만한 헤드를 지우면 외삽 곡선이 어떻게 되는가? 먼 조회를 어느 헤드가 담당하는지 확인하라.
3. (d)의 보간 계수를 $\sqrt{T_{\rm tr}/T}$로 완화하면 곡선이 어디에 놓이는가? 압축(외삽)과 분해능(내삽)의 교환을 그려 보라.
4. `learned`의 수선을 "재사용" 대신 무작위 벡터로 바꾸면 (a)(b)가 어떻게 변하는가? 유도 회로가 위치 정보에 얼마나 기대는지의 간접 측정이다.

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| 재방문 확률 | 1절 | 0.55 | 조회 거리 분포 |
| `T_TR` | 1절 | 48 | 학습 창 |
| `T_EVALS` | 2절 | ≤384 | 외삽 배율 (최대 8배) |
| `V` | 1절 | 64 | 후보 밀도와 (c)의 이동 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")